# Parse a messy date column

Real-world date columns rarely use a single format. You'll see ISO dates mixed with `DD/MM/YYYY`, English month names, Excel serial numbers, empty strings, and the occasional `"N/A"`. This recipe shows the standard playbook for cleaning a column like that into proper `date` objects.

## The data

Pretend we pulled this out of a CSV. It's deliberately awful:

In [ ]:
raw = [
    "2026-04-21",
    "21/04/2026",
    "21-Apr-2026",
    "April 21, 2026",
    "",              # missing
    "N/A",           # sentinel
    "not a date",    # junk
    "2026/04/21",    # yet another
]

## Step 1: multi-format parser

Try each format in turn; return the first one that works. This is the core loop.

In [ ]:
from datetime import datetime

FORMATS = (
    "%Y-%m-%d",
    "%d/%m/%Y",
    "%d-%b-%Y",
    "%B %d, %Y",
    "%Y/%m/%d",
)

def try_parse(s):
    """Return a date or None if no format matches."""
    for fmt in FORMATS:
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    return None

for s in raw:
    print(f"{s!r:25} -> {try_parse(s)}")

`try_parse` returns `None` for strings no format matches — including empties, sentinels, and junk. That's usually the right shape for downstream processing: `None` collapses cleanly into `NaT` in pandas, `NULL` in SQL, missing in most aggregations.

## Step 2: handle sentinels explicitly

If `"N/A"` or `""` should be treated as "known missing" rather than "unparseable", filter them out up front. Makes the distinction between genuinely bad data and expected gaps clearer.

In [ ]:
MISSING = {"", "N/A", "n/a", "null", "NULL"}

def parse_date(s):
    if s is None or s.strip() in MISSING:
        return None
    result = try_parse(s.strip())
    if result is None:
        raise ValueError(f"unparseable date: {s!r}")
    return result

for s in raw:
    try:
        print(f"{s!r:25} -> {parse_date(s)}")
    except ValueError as e:
        print(f"{s!r:25} -> ERROR: {e}")

Three categories now: parsed date, explicit `None`, or `ValueError`. Raise on the third so the bad rows surface instead of silently becoming missing.

## Step 3: ambiguity

`"04/05/2026"` — is that 4 May (British) or 5 April (American)? `strptime` can't tell you; whichever format you put first wins. Pick the one that matches your data source and commit to it.

If you genuinely have both conventions in the same column, you're stuck: no amount of parsing logic can disambiguate `"04/05/2026"` from the string alone. You'll need a secondary signal (the source system, a column header, the day value being > 12) or to raise and reject.

In [ ]:
# British-first
british = try_parse("04/05/2026")    # 4 May
print("British:", british)

# If you want American-first, change FORMATS to put "%m/%d/%Y" ahead of "%d/%m/%Y".
# There is no way to "auto-detect" — the string alone doesn't carry the information.

## Step 4: in pandas

For pandas, `pd.to_datetime` handles most of this for you — including mixed formats and missing values. The trick is `errors="coerce"` so unparseable rows become `NaT` instead of raising:

In [ ]:
# import pandas as pd
# 
# series = pd.Series(raw)
# parsed = pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)
# print(parsed)

# `format="mixed"` lets each row pick its own format (Python 3.11+ / pandas 2.0+).
# `dayfirst=True` disambiguates DD/MM vs MM/DD in favour of the British convention.
# `errors="coerce"` turns unparseables into NaT rather than raising.
# Use .isna() to count how many rows failed, and investigate them by hand.

## Tips

- **Log the unparseable rows** — don't just drop them silently. They'll tell you about a format you missed.
- **Normalise before parsing** — `s.strip()`, lower-casing "n/a", etc., so slight input variations don't cause misses.
- **Keep FORMATS narrow** — every format you add is another chance for a row to match the wrong one. Only add formats you've actually seen in the data.
- **Try `fromisoformat` first** — much faster than `strptime` if most of your data is ISO 8601.